In [63]:
# pip install GEOparse

In [64]:
import GEOparse

# Download dataset
gse = GEOparse.get_GEO("GSE18123", destdir="./data", annotate_gpl=False)



31-Mar-2026 14:05:10 DEBUG utils - Directory ./data already exists. Skipping.
31-Mar-2026 14:05:10 INFO GEOparse - File already exist: using local version.
31-Mar-2026 14:05:10 INFO GEOparse - Parsing ./data\GSE18123_family.soft.gz: 
31-Mar-2026 14:05:10 DEBUG GEOparse - DATABASE: GeoMiame
31-Mar-2026 14:05:10 DEBUG GEOparse - SERIES: GSE18123
31-Mar-2026 14:05:10 DEBUG GEOparse - PLATFORM: GPL570
c:\Users\BAPS\Desktop\Autism\Autism_GEO\venv\lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
31-Mar-2026 14:05:12 DEBUG GEOparse - PLATFORM: GPL6244
31-Mar-2026 14:05:14 DEBUG GEOparse - SAMPLE: GSM650510
31-Mar-2026 14:05:14 DEBUG GEOparse - SAMPLE: GSM650512
31-Mar-2026 14:05:14 DEBUG GEOparse - SAMPLE: GSM650513
31-Mar-2026 14:05:15 DEBUG GEOparse - SAMPLE: GSM650514
31-Mar-2026 14:05:15 DEBUG GEOparse - SAMPLE: GSM650515
31-Mar-2026 14

In [65]:
print("Total Samples:", len(gse.gsms))
print("Platforms:", list(gse.gpls.keys()))

Total Samples: 285
Platforms: ['GPL570', 'GPL6244']


In [66]:
# Inspect 2 sample metadata
for i, (gsm, obj) in enumerate(gse.gsms.items()):
    print(gsm)
    print(obj.metadata.keys())
    print("------")
    if i == 2:
        break

GSM650510
dict_keys(['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_laboratory', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_state', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count'])
------
GSM650512
dict_keys(['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_ema

# 🚀 PHASE 2 — Expression Matrix + Platform-wise Split

In [67]:
import pandas as pd

# 1. Create expression matrix (probe-level)
expression_data = gse.pivot_samples(values='VALUE')

print("Expression shape (probes x samples):", expression_data.shape)


# 2. Create metadata dataframe
meta_data = pd.DataFrame({
    gsm_name: gsm.metadata
    for gsm_name, gsm in gse.gsms.items()
}).T

# Extract platform info
meta_data['platform'] = meta_data['platform_id'].apply(lambda x: x[0])

print("\nPlatform counts:")
print(meta_data['platform'].value_counts())


# 3. Split samples by platform
gpl570_samples = meta_data[meta_data['platform'] == 'GPL570'].index
gpl6244_samples = meta_data[meta_data['platform'] == 'GPL6244'].index

expr_570 = expression_data[gpl570_samples]
expr_6244 = expression_data[gpl6244_samples]

print("\nGPL570 shape:", expr_570.shape)
print("GPL6244 shape:", expr_6244.shape)

Expression shape (probes x samples): (87910, 285)

Platform counts:
platform
GPL6244    186
GPL570      99
Name: count, dtype: int64

GPL570 shape: (87910, 99)
GPL6244 shape: (87910, 186)


In [68]:
print(len(gpl570_samples) + len(gpl6244_samples))

285


In [69]:
print(set(expression_data.columns) - set(meta_data.index))

set()


# 🚀 PHASE 3 — Probe → Gene Symbol Mapping

In [70]:
# Extract platform annotation tables
gpl570_annot = gse.gpls['GPL570'].table
gpl6244_annot = gse.gpls['GPL6244'].table

print("GPL570 annotation shape:", gpl570_annot.shape)
print("GPL6244 annotation shape:", gpl6244_annot.shape)


# Keep only required columns
gpl570_map = gpl570_annot[['ID', 'Gene Symbol']].copy()
gpl6244_map = gpl6244_annot[['ID', 'gene_assignment']].copy()

# Drop missing gene symbols
gpl570_map = gpl570_map.dropna()
gpl6244_map = gpl6244_map.dropna()

# Rename the gene_assignment column to Gene Symbol for consistency
gpl6244_map.rename(columns={'gene_assignment': 'Gene Symbol'}, inplace=True)

print("\nAfter cleaning:")
print("GPL570 map:", gpl570_map.shape)
print("GPL6244 map:", gpl6244_map.shape)


# 🔥 IMPORTANT: Filter expression to only valid probes for each platform

expr_570_filtered = expr_570.loc[expr_570.index.isin(gpl570_map['ID'])]
expr_6244_filtered = expr_6244.loc[expr_6244.index.isin(gpl6244_map['ID'])]

print("\nFiltered expression shapes:")
print("GPL570:", expr_570_filtered.shape)
print("GPL6244:", expr_6244_filtered.shape)


# Merge probe → gene symbol
expr_570_merged = expr_570_filtered.merge(
    gpl570_map, left_index=True, right_on='ID'
)

expr_6244_merged = expr_6244_filtered.merge(
    gpl6244_map, left_index=True, right_on='ID'
)

print("\nMerged shapes:")
print("GPL570:", expr_570_merged.shape)
print("GPL6244:", expr_6244_merged.shape)

GPL570 annotation shape: (54675, 16)
GPL6244 annotation shape: (33297, 12)

After cleaning:
GPL570 map: (45782, 2)
GPL6244 map: (33297, 2)

Filtered expression shapes:
GPL570: (45772, 99)
GPL6244: (33297, 186)

Merged shapes:
GPL570: (45772, 101)
GPL6244: (33297, 188)


In [71]:
print("✔ Check 1 — No massive loss")

print(expr_570_filtered.shape[0] > 40000)
print(expr_6244_filtered.shape[0] > 20000)

✔ Check 1 — No massive loss
True
True


In [72]:
print("✔ Check 2 — Gene Symbol exists")

print("Gene Symbol" in expr_570_merged.columns)

✔ Check 2 — Gene Symbol exists
True


In [73]:
print("✔ Check 3 — No duplicate merge explosion")
expr_570_merged.head()

✔ Check 3 — No duplicate merge explosion


,GSM650510,GSM650512,GSM650513,GSM650514,GSM650515,GSM650516,GSM650517,GSM650518,GSM650519,GSM650520,...,GSM650646,GSM650647,GSM650648,GSM650649,GSM650651,GSM650652,GSM650653,GSM650654,ID,Gene Symbol
0,26.65640,107.61809,48.97026,72.65312,80.83264,40.13933,47.43693,57.90003,65.03053,43.14078,...,49.50204,64.66049,110.58686,18.62053,66.15737,76.31541,59.93721,45.99937,1007_s_at,DDR1 /// MIR4640
1,39.00726,135.73965,51.37555,102.65829,78.93955,81.75657,79.74300,74.16394,131.21986,62.78481,...,106.72321,37.18311,45.04578,77.12382,92.94504,137.79575,26.70354,101.07505,1053_at,RFC2
2,296.01724,459.14382,378.86384,618.79730,266.87741,320.33815,685.72845,285.89160,996.76063,231.43584,...,539.54601,75.35680,350.91103,488.36033,252.50309,329.28063,145.30082,276.80480,117_at,HSPA6
3,209.03575,232.08352,190.15020,185.90196,216.47953,172.25500,188.12218,155.99192,172.30327,141.44389,...,206.78336,281.03506,189.41136,294.80112,274.37934,232.94968,204.35111,265.61533,121_at,PAX8
4,63.82730,50.20000,60.23913,22.03061,41.29200,74.49414,16.84841,48.88556,19.70762,17.45545,...,21.37225,61.94335,37.83045,19.08806,43.59119,49.46663,36.18051,16.70463,1255_g_at,GUCA1A


# 🚀 PHASE 4 — Gene-Level Aggregation

In [74]:
# 1. Keep only necessary columns
expr_570_clean = expr_570_merged.drop(columns=['ID'])
expr_6244_clean = expr_6244_merged.drop(columns=['ID'])


# 2. Handle multiple gene symbols (split "///")
expr_570_clean['Gene Symbol'] = expr_570_clean['Gene Symbol'].str.split(' /// ').str[0]
expr_6244_clean['Gene Symbol'] = expr_6244_clean['Gene Symbol'].str.split(' /// ').str[0]


# 3. Remove empty gene symbols (if any)
expr_570_clean = expr_570_clean[expr_570_clean['Gene Symbol'] != '']
expr_6244_clean = expr_6244_clean[expr_6244_clean['Gene Symbol'] != '']


# 4. Aggregate probes → gene (mean)
expr_570_gene = expr_570_clean.groupby('Gene Symbol').mean()
expr_6244_gene = expr_6244_clean.groupby('Gene Symbol').mean()


print("Gene-level shapes:")
print("GPL570:", expr_570_gene.shape)
print("GPL6244:", expr_6244_gene.shape)

Gene-level shapes:
GPL570: (22880, 99)
GPL6244: (23693, 186)


In [75]:
print("✔ Check 1 — No duplicate genes")
print(expr_570_gene.index.duplicated().sum())
print(expr_6244_gene.index.duplicated().sum())

✔ Check 1 — No duplicate genes
0
0


In [76]:
print("✔ Check 2 — Gene count reasonable")
print(len(expr_570_gene))
print(len(expr_6244_gene))

✔ Check 2 — Gene count reasonable
22880
23693


In [77]:
print("✔ Check 3 — Values look realistic")
expr_570_gene.head()

✔ Check 3 — Values look realistic


,GSM650510,GSM650512,GSM650513,GSM650514,GSM650515,GSM650516,GSM650517,GSM650518,GSM650519,GSM650520,...,GSM650644,GSM650645,GSM650646,GSM650647,GSM650648,GSM650649,GSM650651,GSM650652,GSM650653,GSM650654
Gene Symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,41.63259,83.28202,52.959000,44.688750,67.330120,60.513170,44.421540,26.769420,58.695100,46.864560,...,75.627900,63.707620,33.292570,46.106280,54.652120,49.665740,64.701680,69.44312,26.25912,59.140340
A1BG-AS1,8.17440,1.18151,10.394910,9.956910,16.326440,0.197270,7.871850,0.628880,4.194580,0.004840,...,10.395270,0.241240,15.681010,5.025300,7.780800,8.694080,1.798410,0.01785,17.68914,5.513350
A1CF,95.64004,37.20313,69.185405,42.768305,46.413910,60.622755,59.372435,39.065420,26.172005,51.082100,...,30.916605,48.519265,46.315170,95.476055,52.441430,73.571020,52.925425,36.26605,106.74610,63.756045
A2M,137.95764,81.00714,142.215430,62.315015,90.522845,96.440085,88.301515,54.964955,66.736025,55.187155,...,74.264400,124.375640,72.304515,171.866215,117.637435,111.514715,139.480970,79.03712,170.45382,139.043470
A2M-AS1,9.69171,25.73246,32.337940,31.629130,77.091910,26.903060,34.035480,140.055140,177.304150,118.263660,...,12.995710,34.047090,30.125770,98.480480,252.059920,32.989740,58.633670,95.90285,129.58923,41.255640


In [78]:
expr_6244_gene.head()

,GSM650655,GSM650656,GSM650657,GSM650658,GSM650659,GSM650660,GSM650661,GSM650662,GSM650663,GSM650664,...,GSM650944,GSM650946,GSM650950,GSM650951,GSM650954,GSM650959,GSM650968,GSM650969,GSM650970,GSM650975
Gene Symbol,,,,,,,,,,,,,,,,,,,,,
---,520.533288,538.240634,515.11235,557.767284,536.58291,533.360848,466.986164,537.62047,465.096335,545.353198,...,479.333716,467.761837,516.715599,488.341008,480.632734,512.429114,512.520389,522.854658,505.42821,490.370997
AB014731 // DENR // density-regulated protein // 12q24.31 // 8562,529.517700,626.125640,205.28739,368.173890,471.88618,504.924490,352.867110,395.77933,362.170850,452.737550,...,248.767780,279.743290,274.916900,256.599800,305.274380,328.666240,258.888080,280.499070,445.52838,258.298380
AB014771 // MOP-1 // MOP-1 // 4q21.22 // 643616,81.499590,36.487070,51.48448,90.727050,92.45871,103.367770,70.665680,87.91672,85.236900,90.859330,...,79.726160,57.636300,95.071150,61.422470,68.538950,54.922950,59.961520,65.027970,62.79637,62.344770
AB015046 // XYLB // xylulokinase homolog (H. influenzae) // 3p22-p21.3 // 9942,90.335600,101.991660,81.21358,105.775900,113.09757,98.637490,70.853750,77.23314,91.914870,103.353980,...,119.637380,103.169610,147.457990,131.466150,107.805140,116.957300,136.409900,138.554660,158.87230,129.454200
AB017006 // PMS2L2 // postmeiotic segregation increased 2-like 2 pseudogene // 7q11.23 // 5380,923.894560,1147.330690,945.77713,1092.606140,849.86216,1058.269240,560.610340,741.77719,488.312880,772.001450,...,765.600280,793.871470,980.883930,967.089090,1032.529560,752.649580,754.348050,781.635040,949.20120,1010.763130


# 🚀 PHASE 5 — Clean Gene Symbols + Cross-Platform Integration

In [79]:
# 1. Function to properly extract gene symbols
def extract_gene_symbol(symbol):
    if pd.isna(symbol):
        return None

    # Split by '//' and take second part if available
    parts = [p.strip() for p in symbol.split('//')]

    if len(parts) >= 2:
        return parts[1]  # actual gene symbol
    else:
        return parts[0]


# Apply extraction
expr_570_gene.index = expr_570_gene.index.map(extract_gene_symbol)
expr_6244_gene.index = expr_6244_gene.index.map(extract_gene_symbol)


# 2. Remove invalid entries
def clean_genes_v2(df):
    df = df.copy()

    # Remove invalid
    df = df[~df.index.isin(['---', '', None])]

    # Remove NaN
    df = df[~df.index.isna()]

    return df


# 🔥 Re-aggregate to remove duplicate gene symbols

# Apply clean_genes_v2 to the dataframes before re-aggregation
expr_570_gene_cleaned = clean_genes_v2(expr_570_gene)
expr_6244_gene_cleaned = clean_genes_v2(expr_6244_gene)

# Now, re-aggregate by the cleaned gene symbol index
expr_570_clean2 = expr_570_gene_cleaned.groupby(expr_570_gene_cleaned.index).mean()
expr_6244_clean2 = expr_6244_gene_cleaned.groupby(expr_6244_gene_cleaned.index).mean()

print("After removing duplicates:")
print("GPL570:", expr_570_clean2.shape)
print("GPL6244:", expr_6244_clean2.shape)


# Now find common genes again
common_genes = expr_570_clean2.index.intersection(expr_6244_clean2.index)

print("\nCommon genes count:", len(common_genes))


# Align datasets
expr_570_common = expr_570_clean2.loc[common_genes]
expr_6244_common = expr_6244_clean2.loc[common_genes]


# Concatenate safely
combined_expr = pd.concat([expr_570_common, expr_6244_common], axis=1)

print("\nFinal combined shape:", combined_expr.shape)


After removing duplicates:
GPL570: (22880, 99)
GPL6244: (23307, 186)

Common genes count: 18036

Final combined shape: (18036, 285)


In [80]:
# 1. Check index uniqueness
print(expr_570_clean2.index.is_unique)
print(expr_6244_clean2.index.is_unique)

# Must be True True


# 2. Sample count
print(combined_expr.shape[1])  # must be 285


# 3. Gene alignment
print(expr_570_common.index.equals(expr_6244_common.index))  # must be True

True
True
285
True


# 🚀 PHASE 6 — Transpose + Label Alignment

In [81]:
# 1. Transpose (samples as rows)
X = combined_expr.T

print("Transposed shape (samples x genes):", X.shape)


# 2. Build labels from metadata
def extract_label(characteristics):
    text = " ".join(characteristics).lower()

    if "autism" in text or "asperger" in text or "pdd" in text:
        return 1
    elif "control" in text:
        return 0
    else:
        return None  # unknown


meta_data['label'] = meta_data['characteristics_ch1'].apply(extract_label)


# 3. Align labels with X
y = meta_data.loc[X.index, 'label']

print("\nLabel distribution:")
print(y.value_counts(dropna=False))


# 4. Remove samples with unknown labels (if any)
valid_idx = y.dropna().index

X = X.loc[valid_idx]
y = y.loc[valid_idx]

print("\nAfter removing unknown labels:")
print("X shape:", X.shape)
print("y shape:", y.shape)

Transposed shape (samples x genes): (285, 18036)

Label distribution:
label
1    170
0    115
Name: count, dtype: int64

After removing unknown labels:
X shape: (285, 18036)
y shape: (285,)


In [108]:
X.head(1)

Gene Symbol,A1BG,A1CF,A2M,A2ML1,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL2,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM650510,41.63259,95.64004,137.95764,37.034905,31.90844,17.88647,45.29633,17.54859,0.02602,3.32522,...,48.976425,14.58944,32.558653,19.40356,37.706935,9.73246,90.540157,798.485975,271.942207,31.20494


In [109]:
y.head(1)

GSM650510    1
Name: label, dtype: int64

In [82]:
print("CRITICAL VALIDATION CHECKS")

print(all(X.index == y.index))

print(y.isna().sum())

print(y.value_counts(normalize=True))



CRITICAL VALIDATION CHECKS
True
0
label
1    0.596491
0    0.403509
Name: proportion, dtype: float64


# 🚀 PHASE 7 — Normalization + Batch Effect Handling

In [83]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Log2 transformation (VERY IMPORTANT for microarray)
X_log = np.log2(X + 1)

print("After log transform:")
print(X_log.iloc[:2, :5])


# 2. Z-score normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("\nScaled shape:", X_scaled.shape)


print("After Z-score normalization")
print(X_scaled[:5, :5])

After log transform:
Gene Symbol      A1BG      A1CF       A2M     A2ML1    A4GALT
GSM650510    5.413885  6.594549  7.118501  5.249252  5.040386
GSM650512    6.397153  5.255619  6.357678  4.077491  4.869053

Scaled shape: (285, 18036)
After Z-score normalization
[[-1.57245801  0.42039289  0.48781655 -0.10880352 -1.06344445]
 [-0.56421853 -1.7801168  -1.20863277 -1.84328348 -1.18287119]
 [-1.22391875 -0.33799256  0.58490398 -1.15554829 -0.64279849]
 [-1.47003911 -1.45767246 -2.04078089 -0.63706163 -1.18363059]
 [-0.87460927 -1.26797525 -0.85547913 -1.23615841 -1.27558209]]


In [84]:
print("CRITICAL VALIDATION CHECKS")

# Convert back to DataFrame for inspection
import pandas as pd

X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

# 1. Mean ~ 0
print(X_scaled_df.mean().mean())

# 2. Std ~ 1
print(X_scaled_df.std().mean())

# 3. No NaNs
print(X_scaled_df.isna().sum().sum())

CRITICAL VALIDATION CHECKS
-2.0137661972806116e-17
1.0017590163110905
0


In [85]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Train-test split (IMPORTANT: stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


# # 2. Train model
# model = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=None,
#     max_features='sqrt',
#     random_state=42,
#     min_samples_leaf=2,
#     min_samples_split=5,
#     n_jobs=-1
# )

# model.fit(X_train, y_train)


# # 3. Predictions
# y_pred = model.predict(X_test)


# # 4. Evaluation
# print("\nAccuracy:", accuracy_score(y_test, y_pred))

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

Train shape: (228, 18036)
Test shape: (57, 18036)


In [86]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


# # 3. Train ANN model
# # Convert y_train and y_test to numpy arrays for Keras
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()



## 🚀 PHASE 8 — Training Final ANN Model with Best Hyperparameters

In [95]:
# Define the ANN model with additional hidden layers and Dropout
def create_model(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        # Dropout(0.3),
        # Dense(4096, activation='relu'),
        # Dropout(0.3),
        # Dense(2048, activation='relu'),
        # Dropout(0.3),
        # Dense(1024, activation='relu'),
        # Dropout(0.2),
        # Dense(512, activation='relu'),
        # Dropout(0.1),
        # Dense(256, activation='relu'),
        # Dense(128, activation='relu'),
        # Dropout(0.1),
        # Dense(2, activation='relu'),
        Dense(1, activation='sigmoid') # Sigmoid for binary classification
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


# Create the final ANN model with the best hyperparameters
final_model = create_model(X_train.shape[1])

print("\n--- Final Model Summary ---")
final_model.summary()

best_result = {'epochs':25,'batch_size':64,'validation_split':0.1}
# Train the final model with the best hyperparameters
history_final = final_model.fit(
    X_train, y_train_np,
    epochs=int(best_result['epochs']),
    batch_size=int(best_result['batch_size']),
    validation_split=best_result['validation_split'],
    verbose=1 # Show progress during training
)

# Make predictions with the final model
y_pred_prob_final = final_model.predict(X_test)
y_pred_final = (y_pred_prob_final > 0.5).astype(int)





--- Final Model Summary ---


c:\Users\BAPS\Desktop\Autism\Autism_GEO\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 64)             │     1,154,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,154,433 (4.40 MB)

 Trainable params: 1,154,433 (4.40 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 198ms/step - accuracy: 0.5073 - loss: 10.9987 - val_accuracy: 0.5652 - val_loss: 25.0773
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5463 - loss: 9.5994 - val_accuracy: 0.7391 - val_loss: 9.9277
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.5854 - loss: 15.9143 - val_accuracy: 0.7391 - val_loss: 13.6080
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.6732 - loss: 8.9860 - val_accuracy: 0.6087 - val_loss: 18.6681
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.6293 - loss: 10.6924 - val_accuracy: 0.5652 - val_loss: 20.5925
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.7268 - loss: 6.3029 - val_accuracy: 0.6522 - val_loss: 10.3269
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.6732 - loss: 8.5010 - val_accuracy: 0.6522 - val_loss: 11.7813
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.7171 - loss: 7.1676 - val_accuracy: 0.6957 - val_los

In [96]:
# Evaluate the final model
print("\n--- Final Model Evaluation ---")
print("Accuracy:", accuracy_score(y_test_np, y_pred_final))
print("\nClassification Report:")
print(classification_report(y_test_np, y_pred_final))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_np, y_pred_final))


--- Final Model Evaluation ---
Accuracy: 0.8070175438596491

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.78      0.77        23
           1       0.85      0.82      0.84        34

    accuracy                           0.81        57
   macro avg       0.80      0.80      0.80        57
weighted avg       0.81      0.81      0.81        57


Confusion Matrix:
[[18  5]
 [ 6 28]]


# SAVE MODEL + PREPROCESSING STEPS

In [99]:
import pickle

# 1. Save ANN model
final_model.save("autism_ann_model.h5")

# 2. Save preprocessing objects
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# If you used feature selection
try:
    with open("selector.pkl", "wb") as f:
        pickle.dump(selector, f)
except:
    print("Selector not found, skipping...")

# Save feature columns (VERY IMPORTANT)
with open("feature_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("✅ Model and preprocessing saved successfully!")

Selector not found, skipping...
✅ Model and preprocessing saved successfully!


# LOAD MODEL FOR PREDICTION

In [100]:
from tensorflow.keras.models import load_model
import pickle

# Load model
model = load_model("autism_ann_model.h5")

# Load preprocessing
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

# Load selector if exists
try:
    with open("selector.pkl", "rb") as f:
        selector = pickle.load(f)
except:
    selector = None

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


# SINGLE SAMPLE CSV PREDICTION

In [ ]:
import pandas as pd
import numpy as np

def predict_sample(csv_path):
    
    # 1. Load sample
    sample_df = pd.read_csv(csv_path)
    
    # Expecting: Gene Symbol, Expression
    sample_df.columns = ["Gene", "Value"]
    
    # 2. Convert to dictionary
    sample_dict = dict(zip(sample_df["Gene"], sample_df["Value"]))
    
    # 3. Align with training features
    sample_vector = []
    
    for gene in feature_columns:
        sample_vector.append(sample_dict.get(gene, 0))  # fill missing genes with 0
    
    sample_vector = np.array(sample_vector).reshape(1, -1)
    
    # 4. Log transform (same as training)
    sample_vector = np.log2(sample_vector + 1)
    
    # 5. Scale
    sample_vector = scaler.transform(sample_vector)
    
    # 6. Feature selection (if used)
    # if selector:
    #     sample_vector = selector.transform(sample_vector)
    
    # 7. Predict
    prediction = model.predict(sample_vector)
    
    # Convert probability → class
    predicted_class = (prediction > 0.5).astype(int)[0][0]
    
    print("\n🔍 Prediction Probability:", prediction[0][0])
    
    if predicted_class == 1:
        print("🧠 Result: AUTISM")
    else:
        print("🧠 Result: CONTROL")
    
    return predicted_class

In [125]:
X.head(10)

Gene Symbol,A1BG,A1CF,A2M,A2ML1,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL2,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM650510,41.63259,95.640040,137.957640,37.034905,31.90844,17.88647,45.29633,17.54859,0.02602,3.32522,...,48.976425,14.58944,32.558653,19.403560,37.706935,9.73246,90.540157,798.485975,271.942207,31.204940
GSM650512,83.28202,37.203130,81.007140,15.882900,28.22341,24.10993,66.79974,102.38333,6.63567,2.58592,...,125.516175,114.41336,29.698750,60.779100,95.003493,1.84859,305.102487,1022.778865,254.717260,158.735530
GSM650513,52.95900,69.185405,142.215430,22.297505,49.00002,18.52839,51.23941,37.13484,5.13304,0.02840,...,45.013960,22.81288,19.520257,30.196130,92.381738,7.71809,202.538620,707.486985,341.249793,47.567460
GSM650514,44.68875,42.768305,62.315015,28.699715,28.20135,14.28094,35.27751,68.47804,2.95852,3.97667,...,88.122150,82.61474,25.445117,58.672835,108.238012,12.20726,342.711103,856.898565,268.922863,130.310685
GSM650515,67.33012,46.413910,90.522845,21.434480,25.64969,16.16558,59.70023,92.80730,10.10688,3.82218,...,84.270565,34.40494,26.058877,56.821985,108.401883,8.46415,309.137360,759.016425,320.187483,110.856830
GSM650516,60.51317,60.622755,96.440085,23.368040,27.83805,28.19630,53.30302,49.21835,0.00212,8.90778,...,71.055910,41.44445,26.000773,41.438915,89.886213,7.86305,230.600267,780.765135,304.505493,83.165085
GSM650517,44.42154,59.372435,88.301515,18.971925,14.63649,17.90235,76.85973,46.73850,4.75357,8.13893,...,75.361850,35.78393,18.377007,45.747870,125.313958,6.60387,312.511813,939.762930,289.604057,91.090515
GSM650518,26.76942,39.065420,54.964955,14.569970,27.29699,11.77577,73.85845,88.90493,7.20402,0.02082,...,89.049655,45.51507,19.547380,44.855585,131.537537,9.21605,314.228910,633.710205,305.160403,108.470675
GSM650519,58.69510,26.172005,66.736025,10.406435,27.40028,20.06977,56.17572,85.12756,5.69794,2.66903,...,117.621025,247.80964,29.534500,52.466520,167.643130,7.73003,377.777233,974.900610,280.125727,146.081040
GSM650520,46.86456,51.082100,55.187155,26.289585,17.10492,8.27650,53.11450,53.19196,3.52249,0.00338,...,80.224435,44.93666,22.665300,53.020405,124.076502,11.69462,259.726963,760.832645,288.047563,114.540680


In [118]:
y.head(10)

GSM650510    1
GSM650512    1
GSM650513    1
GSM650514    1
GSM650515    1
GSM650516    1
GSM650517    1
GSM650518    1
GSM650519    1
GSM650520    1
Name: label, dtype: int64

In [126]:
# Create one sample from existing dataset
sample = X.iloc[0]  # use the 1st sample for testing

print(sample.head())

sample_df = pd.DataFrame({
    "Gene": sample.index,
    "Value": sample.values
})

sample_df.to_csv("test_sample.csv", index=False)

# Predict
predict_sample("test_sample.csv")

Gene Symbol
A1BG       41.632590
A1CF       95.640040
A2M       137.957640
A2ML1      37.034905
A4GALT     31.908440
Name: GSM650510, dtype: float64
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step

🔍 Prediction Probability: 5.470009e-37
🧠 Result: CONTROL


c:\Users\BAPS\Desktop\Autism\Autism_GEO\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


np.int64(0)

# Save X and y into the CSV file

In [127]:
import pandas as pd

# Convert X to DataFrame if it's not already
if not isinstance(X, pd.DataFrame):
    X_df = pd.DataFrame(X, columns=feature_columns)
else:
    X_df = X.copy()

# Add target column
X_df['label'] = y.values

# Save to CSV
X_df.to_csv("autism_dataset_final.csv", index=True)

print("✅ Dataset saved as autism_dataset_final.csv")
print("Shape:", X_df.shape)

✅ Dataset saved as autism_dataset_final.csv
Shape: (285, 18037)
